# DS Week 02 — Uncertainty-Aware Turbofan Remaining Useful Life Prognostics

**Domain:** Manufacturing / Prognostics & Health Management  
**Task:** RUL regression with state-aware uncertainty  
**Validation:** engine-group-aware, no unit leakage  
**Hardware:** Ryzen 7 / 16 GB RAM / GTX 1650 Ti 4 GB


## 1. Business / scientific framing
Estimate cycles remaining before failure. Optimistic RUL can delay maintenance and raise failure risk; conservative RUL can cause premature service. Questions: do leakage-safe degradation features beat age baselines, does a train-derived health state help, and how trustworthy are prediction intervals under sensor perturbations?


## 2. Dataset provenance and research connection
NASA PCoE C-MAPSS contains run-to-failure turbofan trajectories. See DATASET.md. The research adaptation is inspired by Belaunzaran et al. (2026): we reproduce only the manageable state-aware idea using a health indicator, healthy/degraded regressors and quantile uncertainty; paper claims and project results remain separate.


In [ ]:
from pathlib import Path
import json, random, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from src.features import *
from src.metrics import *
SEED=42; random.seed(SEED); np.random.seed(SEED)
DATA_PATH=Path('data/raw/train_FD001.txt'); ART=Path('artifacts'); ART.mkdir(exist_ok=True)


## 3. Load data and define target
Training RUL is max engine cycle minus current cycle, capped at 125 for early-life ambiguity. If NASA FD001 is absent, execute a deterministic schema-compatible synthetic fleet; synthetic scores are smoke results only.


In [ ]:
raw=load_cmapss_fd001(DATA_PATH) if DATA_PATH.exists() else make_synthetic_fleet(seed=SEED)
DATA_MODE='NASA_CMAPSS_FD001' if DATA_PATH.exists() else 'SYNTHETIC_SMOKE'
print(DATA_MODE, raw.shape, raw.unit.nunique())


## 4. Leakage audit and validation design
Never random-split rows; an engine trajectory must exist in only one partition. No centered windows, no future max-cycle production feature, and all health/normalization statistics are fit on training engines only.


In [ ]:
g=raw.unit.to_numpy(); a,b=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED).split(raw,groups=g)); train_full,test=raw.iloc[a].copy(),raw.iloc[b].copy()
a,b=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED+1).split(train_full,groups=train_full.unit)); train,val=train_full.iloc[a].copy(),train_full.iloc[b].copy()
assert set(train.unit).isdisjoint(val.unit) and set(train.unit).isdisjoint(test.unit)


## 5. EDA, data quality and domain feature engineering
Inspect missingness, engine-life/RUL distributions and low-variance sensors. Build trailing means/stds, lag-1 and lag-window deltas, plus a multivariate health indicator learned from nominal early cycles. Trailing windows are inference-valid because they use only current/past observations.


In [ ]:
sensors=select_informative_sensors(train)
train_h,[val_h,test_h],_,_=add_health_indicator(train,[val,test],sensors)
train_f,val_f,test_f=[add_temporal_features(x,sensors) for x in [train_h,val_h,test_h]]
train_f,[val_f,test_f],HEALTH_THRESHOLD=assign_regime_from_train(train_f,[val_f,test_f])
features=[c for c in production_feature_columns(train_f) if c!='regime']
print('features',len(features),'health threshold',HEALTH_THRESHOLD)


## 6. Baselines, competitive models and hyperparameter strategy
Compare age heuristic, regularized Ridge and nonlinear HistGradientBoosting. Use constrained local hyperparameters; any future search must use engine-group-aware folds rather than ordinary row K-fold.


In [ ]:
median_life=float(train.groupby('unit').cycle.max().median()); age=lambda d:np.clip(median_life-d.cycle.to_numpy(),0,125)
ridge=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=10.)).fit(train_f[features],train_f.rul)
hgb=HistGradientBoostingRegressor(max_iter=180,learning_rate=.06,max_leaf_nodes=24,l2_regularization=1.,min_samples_leaf=25,random_state=SEED).fit(train_f[features],train_f.rul)
comparison=[]
for name,p in [('age_baseline',age(val_f)),('ridge',ridge.predict(val_f[features])),('hist_gbr',hgb.predict(val_f[features]))]: comparison.append({'model':name,**regression_metrics(val_f.rul,p)})
pd.DataFrame(comparison).sort_values('rmse')


## 7. State-aware research adaptation
Fit separate healthy/degraded models using a health threshold learned from train only. This approximates the bifurcated research idea without claiming to reproduce its neural/survival architecture.


In [ ]:
hm=HistGradientBoostingRegressor(max_iter=150,learning_rate=.06,max_leaf_nodes=20,min_samples_leaf=20,random_state=52); dm=HistGradientBoostingRegressor(max_iter=180,learning_rate=.05,max_leaf_nodes=24,min_samples_leaf=18,random_state=53)
hm.fit(train_f.loc[train_f.regime==0,features],train_f.loc[train_f.regime==0,'rul']); dm.fit(train_f.loc[train_f.regime==1,features],train_f.loc[train_f.regime==1,'rul'])
def regime_predict(d):
    out=np.zeros(len(d)); m=d.regime.to_numpy()==0
    if m.any(): out[m]=hm.predict(d.loc[m,features])
    if (~m).any(): out[~m]=dm.predict(d.loc[~m,features])
    return out
comparison.append({'model':'regime_aware_hgb',**regression_metrics(val_f.rul,regime_predict(val_f))})


## 8. Quantile uncertainty
Fit P10/P50/P90 GradientBoosting quantiles. Evaluate interval coverage and width; coverage without sharpness is not sufficient for a maintenance decision.


In [ ]:
med=train_f[features].median(); Xtr=train_f[features].fillna(med); Xv=val_f[features].fillna(med)
qm={}
for a in [.1,.5,.9]: qm[a]=GradientBoostingRegressor(loss='quantile',alpha=a,n_estimators=120,learning_rate=.05,max_depth=3,min_samples_leaf=20,random_state=SEED+int(a*100)).fit(Xtr,train_f.rul)
p10,p50,p90=[qm[a].predict(Xv) for a in [.1,.5,.9]]
interval_val=interval_metrics(val_f.rul,p10,p90); interval_val


## 9. Untouched test, lifecycle slices and explainability
Select on validation RMSE/asymmetric cost, then evaluate untouched engines. Report early/mid/late lifecycle metrics and optimistic-error rate. Compute held-out permutation importance rather than relying on impurity importance.


In [ ]:
cmp=pd.DataFrame(comparison).sort_values(['rmse','asymmetric_cost']); best=cmp.iloc[0].model
preds={'age_baseline':age(test_f),'ridge':ridge.predict(test_f[features]),'hist_gbr':hgb.predict(test_f[features]),'regime_aware_hgb':regime_predict(test_f)}; pred=preds[best]
test_metrics=regression_metrics(test_f.rul,pred)
slices=[]
for label,mask in [('early',test_f.rul>80),('mid',(test_f.rul>30)&(test_f.rul<=80)),('late',test_f.rul<=30)]:
    if mask.sum(): slices.append({'slice':label,**regression_metrics(test_f.loc[mask,'rul'],pred[mask.to_numpy()]),'optimistic_rate':float(np.mean(pred[mask.to_numpy()]>test_f.loc[mask,'rul']))})
slice_df=pd.DataFrame(slices); print(best,test_metrics); display(slice_df)
perm=permutation_importance(hgb,val_f[features],val_f.rul,scoring='neg_root_mean_squared_error',n_repeats=5,random_state=SEED)
pd.DataFrame({'feature':features,'importance':perm.importances_mean}).sort_values('importance',ascending=False).head(15)


## 10. Robustness / sensitivity
Stress the selected model without retraining using measurement noise and gradual sensor drift. Compare clean and perturbed RMSE/MAE/asymmetric cost to estimate deployment fragility.


In [ ]:
rng=np.random.default_rng(SEED); rows=[]
def eval_scenario(name,d):
    pp=regime_predict(d) if best=='regime_aware_hgb' else (hgb.predict(d[features]) if best=='hist_gbr' else ridge.predict(d[features]) if best=='ridge' else age(d)); rows.append({'scenario':name,**regression_metrics(d.rul,pp)})
eval_scenario('clean',test_f.copy()); noise=test_f.copy()
for c in sensors[:6]: noise[c]+=rng.normal(0,float(train_f[c].std())*.08,len(noise))
eval_scenario('sensor_noise_8pct_std',noise); drift=test_f.copy(); progress=drift.groupby('unit').cumcount()/drift.groupby('unit').unit.transform('size').clip(lower=1)
for c in sensors[:4]: drift[c]+=progress*float(train_f[c].std())*.20
eval_scenario('gradual_sensor_drift',drift); robust_df=pd.DataFrame(rows); robust_df


## 11. Experiment comparison, conclusions and limitations
Promotion should consider overall error, late-life optimistic errors, asymmetric risk cost, interval coverage/width and robustness—not RMSE alone. Synthetic fallback results validate execution only. FD001 is controlled; FD002/FD004 add operating-condition shift. Quantile intervals are not automatically calibrated under domain shift, and real fleets commonly contain censoring/interventions.


In [ ]:
cmp.to_csv(ART/'model_comparison.csv',index=False); slice_df.to_csv(ART/'slice_metrics.csv',index=False); robust_df.to_csv(ART/'robustness.csv',index=False)
payload={'data_mode':DATA_MODE,'selected_model':best,'validation_interval':interval_val,'test_metrics':test_metrics,'health_threshold':HEALTH_THRESHOLD,'n_features':len(features)}
(ART/'metrics.json').write_text(json.dumps(payload,indent=2)); payload


## 12. Production / research follow-ups
1. Run NASA FD001 and then FD002/FD004 with condition-aware normalization. 2. Compare compact 1D CNN/LSTM. 3. Add survival analysis for censored fleets. 4. Model heteroscedastic uncertainty. 5. Test uncertainty-aware latent health indicators. 6. Evaluate cross-condition/domain adaptation. 7. Explore FedCMAPSS-style federated fleet learning. 8. Optimize a maintenance policy directly for downtime and premature-maintenance cost.
